# Reproducing Valero et al. (2022) from dandiset 000568

Paper. Valero M, Zutshi I, Yoon E, Buzsaki G (2022). Probing subthreshold dynamics of hippocampal
neurons by pulsed optogenetics. Science 375, 570-574.
[doi:10.1126/science.abm1891](https://doi.org/10.1126/science.abm1891). This attempt used the full
text of the NIH author manuscript in PubMed Central (PMC9632609): the main report and its figure
legends. The Materials and Methods and figures S1 to S15 are in the supplement, which could not be
obtained (PMC's copy is behind a captcha and the publisher's is not open access). Claims about
supplementary figures are taken from the main text's statements of them, and method details come
from the code named below.

Dandiset. [DANDI:000568](https://dandiarchive.org/dandiset/000568/0.230705.1633), version
0.230705.1633, streamed. The paper cites no DANDI version; its data statement points to the Buzsaki
Lab Databank, whose page lists this dandiset for the paper (its other copy is on Globus, which needs
an account and has no pinned identifier, so it was not used). This is the only published version.
The draft holds the same 138 assets (paths, sizes and asset ids, printed below), so nothing the
analysis uses differs.

What the dandiset holds. 28 sessions from 4 mice, each deposited twice (a full file and a
no-raw-data file with the same units). Units carry spike times only, with no cell type. There are
epochs, ripple intervals and linearized position. Laps and uLED pulse times are in 2 sessions only
(fCamk1 sess9 and sess12), so every claim that needs light responses or place fields is tested on
these two sessions of one mouse.

Code read. The paper names buzcode (github.com/buzsakilab/buzcode, commit e615663): the lab's
general toolbox, with place-field and ripple functions but no script for this paper's figures. Its
place-field defaults are used here. The first author's later lab code, HippoCookBook
(github.com/valegarman/HippoCookBook, commit 0cbded3), is not cited by the paper. It holds the
light-response test (getuLEDResponse.m) used here. Whether either matches the paper's Methods could
not be checked, as the Methods were not available.

Outside inputs. None.

| Claim | Figure | Tested on |
|---|---|---|
| cohort-counts | Results | all 28 sessions |
| fig1c-pulse-protocol | 1C | 2 sessions with pulse times |
| fig1e-responsive-fraction | 1E | 2 sessions |
| fig1i-swr-decrease | 1F-I | 2 sessions |
| figs10-swr-gain-vs-baseline-rate | S10 | 24 sessions with ripples |
| fig2b-theta-trough-gain | 2A-B | 2 sessions |
| figs11-place-cell-fraction | S11 | 2 sessions |
| fig2e-infield-gain | 2C-E | 2 sessions |
| figs12-gain-rate-correlations | S12 | 2 sessions |
| fig3a-place-cells-more-excitable | 3A | 2 sessions |
| fig3c-unmasked-fields | 3B-C | 2 sessions |
| fig3e-ghost-fields | 3E | 2 sessions |
| fig3f-light-stability | 3F | 2 sessions |
| fig3g-spatial-information | 3G | 2 sessions |
| fig4c-swr-cofiring-place-pairs | 4B-C | 2 sessions |
| fig4d-light-unmasked-cofiring | 4C-D | 2 sessions |
| fig1j-light-intensity | 1J, 2F, 3D | blocked: data missing (no intensity levels) |
| fig1km-intracellular | 1K-M | blocked: data missing |
| figs9-excitatory-vs-inhibitory-swr | S9 | blocked: metadata insufficient (no cell types) |
| fig3h-position-decoding | 3H | blocked: methods underspecified |
| fig4ef-assemblies | 4E-F | blocked: methods underspecified |
| fig4gh-replay | 4G-H | blocked: methods underspecified |

The run takes about five minutes, most of it streaming the files.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import valero as v

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
assets = v.assets()
small = [a for a in assets if "no-raw-data" in a["path"]]
full = {v.session_key(a["path"]): a for a in assets if "no-raw-data" not in a["path"]}
sessions = v.load_sessions([a["url"] for a in small])
for a, s in zip(small, sessions):
    s["key"] = v.session_key(a["path"])
    s["full_url"] = full[s["key"]]["url"] if s["key"] in full else None
from dandi.dandiapi import DandiAPIClient
with DandiAPIClient() as client:
    listing = {ver: {(a.path, a.size, a.identifier) for a in client.get_dandiset(v.DANDISET_ID, ver).get_assets()}
               for ver in (v.DANDISET_VERSION, "draft")}
    published = [x.identifier for x in client.get_dandiset(v.DANDISET_ID).get_versions() if x.identifier != "draft"]
print("Published versions:", published, "; assets in", v.DANDISET_VERSION, len(listing[v.DANDISET_VERSION]),
      "; in the draft", len(listing["draft"]), "; same paths, sizes and asset ids:", listing[v.DANDISET_VERSION] == listing["draft"])
print(f"{len(assets)} NWB files in dandiset {v.DANDISET_ID} version {v.DANDISET_VERSION}; "
      f"loaded {len(sessions)} no-raw-data files in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
# Every file, paired by session; a pair holds the same units, spikes, ripples and position
# when these fingerprints agree.
from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(16) as ex:
    prints = pd.DataFrame(list(ex.map(v.file_fingerprint, assets)))
prints["kind"] = np.where(prints.path.str.contains("no-raw-data"), "no_raw", "full")
prints["size_GB"] = [round(a["size"] / 1e9, 3) for a in assets]
cols = ["n_units", "n_spikes", "spike_index_sum", "spike_head_sum", "spike_tail_sum", "n_ripples",
        "ripple_sum", "n_position", "position_sum"]
# Files are paired by the session_id they hold (without "_no_raw_data"), not by their names.
pairs = prints.pivot(index="pair_key", columns="kind",
                     values=cols + ["size_GB", "has_raw", "has_lfp", "raw_s", "lfp_s", "last_epoch_end_s"])
same = pd.Series({k: all(np.isclose(pairs.loc[k, (c, "full")], pairs.loc[k, (c, "no_raw")]) for c in cols)
                  for k in pairs.index}, name="identical_content")
print("Files per kind:", prints.kind.value_counts().to_dict())
print("Sessions (by session_id):", prints.pair_key.nunique(), "; sessions with both files:",
      int(pairs[("n_units", "full")].notna().sum()))
print("Sessions whose two files hold the same units, spikes, ripples and position:", int(same.sum()), "of", len(same))
print("Subjects:", sorted(prints.subject.unique()))
print("session_start_time values:", prints.session_start_time.unique().tolist())
print("\nFiles whose name lacks the session:",
      prints[~prints.path.str.contains("ses-")][["path", "session_id"]].values.tolist())
table = pd.DataFrame({"units": pairs[("n_units", "no_raw")], "spikes": pairs[("n_spikes", "no_raw")],
                      "ripples": pairs[("n_ripples", "no_raw")], "full_GB": pairs[("size_GB", "full")],
                      "session_s": pairs[("last_epoch_end_s", "full")].round(0),
                      "full_raw_s": pairs[("raw_s", "full")].round(0), "full_lfp_s": pairs[("lfp_s", "full")].round(0),
                      "identical": same})
print(table.to_string())
short = table[(table.full_raw_s.notna() & (table.full_raw_s < 0.95 * table.session_s)) |
              (table.full_lfp_s < 0.95 * table.session_s)]
print("\nFull files whose raw or LFP signal covers less than 95% of the session:", short.index.tolist())
print("Full files without a raw signal:", table[table.full_raw_s.isna()].index.tolist())

The dandiset holds 56 NWB files and 82 videos. The NWB files cover 28 sessions: each session is
deposited twice, as a full file and a no-raw-data file. In all 28 the two files hold the same units,
spike times, ripples and position (the fingerprints above agree), so each session is counted once
and read from its no-raw-data file. The full file of fCamk3 sess20 lacks the session in its name;
the session_id inside it identifies it. The full file of fCamk3 sess17 is 28 MB: its raw signal
covers 378 s and its LFP 6850 s of a 34621 s session. Six full files of fCamk1 have the LFP but no
raw signal. In fCamk2 sess4 the signals run 16736 s, past the last epoch at 12781 s.
`session_start_time` is 2016-03-15 in every file, while the session ids date the recordings to 2020
and 2021, so the start time is wrong.

In [ ]:
# Epochs, laps, ripples and pulse series per session; baseline rates of the units.
rows = []
for s in sessions:
    e0 = s["epochs"][0]
    dur = e0["stop"] - e0["start"]
    s["base_rate"] = np.array([((x >= e0["start"]) & (x < e0["stop"])).sum() / dur for x in s["spikes"]])
    s["pyr"] = s["base_rate"] < v.PYR_MAX_RATE_HZ
    paradigms = [e.get("behavioral_paradigm", "") for e in s["epochs"]]
    manip = [e.get("manipulation", "") for e in s["epochs"]]
    s["stim_protocol"] = any("uled" in m.lower() for m in manip)
    rows.append({"session": s["key"].split("_ses-")[1], "subject": s["subject"], "units": len(s["spikes"]),
                 "putative_pyr": int(s["pyr"].sum()), "first_epoch_s": round(dur),
                 "epochs": len(s["epochs"]), "labelled": any(paradigms), "uLED_epochs": s["stim_protocol"],
                 "chemogenetic": any(("CNO" in p) or ("DMSO" in p) for p in paradigms),
                 "laps": len(s["laps"]), "ripples": len(s["ripples"]),
                 "pulse_series": len(s["pulses"]), "position_samples": len(s.get("pos_t", []))})
overview = pd.DataFrame(rows)
print(overview.to_string(index=False))
print("\nSessions per subject:", overview.groupby("subject").size().to_dict())
print("Sessions with pulse times:", overview[overview.pulse_series > 0].session.tolist())
print("Sessions with laps:", overview[overview.laps > 0].session.tolist())
print("Sessions without epoch labels:", overview[~overview.labelled].session.tolist())
print("Sessions with CNO or DMSO epochs:", int(overview.chemogenetic.sum()))
print("\nEpoch names of each kind of session:")
for s in sessions:
    if s["key"].endswith(("fCamk1-200827-sess9", "fCamk3-201029-sess11-cleanned", "fCamk2-201012-sess1")):
        print(" ", s["key"], [(e.get("behavioral_paradigm") or e.get("epoch_name"), e.get("manipulation", "")) for e in s["epochs"]])

There are four subjects, matching the paper's four mice. Pulse times are deposited for 2 sessions
only (fCamk1 sess9 and sess12). The other 20 sessions whose epochs name "uLED random stim" have no
pulse times, so their light responses cannot be computed. Only the same two sessions have laps;
position is present in 24 sessions. Six sessions (all four of fCamk2, fCamk3 sess13 and fCamk5
sess10) name their epochs only by a time stamp, without paradigm or manipulation. 14 sessions have a
second block of epochs named CNO or DMSO, a chemogenetic manipulation that the paper does not
describe. fCamk2 has no position and 0 to 1110 ripples per session.

In [ ]:
# Baseline firing rates, and the putative pyramidal threshold.
rates = np.concatenate([s["base_rate"] for s in sessions])
print("Units:", len(rates), "; baseline rate percentiles 5/25/50/75/95 (Hz):",
      np.percentile(rates, [5, 25, 50, 75, 95]).round(2).tolist(), "; max", rates.max().round(1))
for thr in (2.5, 5, 10, 15):
    print(f"  below {thr:>4} Hz: {(rates < thr).sum()} units ({(rates < thr).mean():.1%})")
isi_viol = []
for s in sessions:
    for x in s["spikes"]:
        if len(x) > 100:
            isi_viol.append(np.mean(np.diff(x) < 0.002))
isi_viol = np.array(isi_viol)
print("Units whose spike times are not sorted:", sum(int(np.any(np.diff(x) < 0)) for s in sessions for x in s["spikes"]))
print("Fraction of ISIs below 2 ms per unit, percentiles 50/90/99:", np.percentile(isi_viol, [50, 90, 99]).round(4).tolist())
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(np.log10(rates + 1e-3), bins=60, color="0.4")
ax.axvline(np.log10(v.PYR_MAX_RATE_HZ), color="C3", lw=1)
ax.set_xlabel("log10 baseline rate (Hz)")
ax.set_ylabel("units")
ax.set_title(f"all {len(rates)} units; red: {v.PYR_MAX_RATE_HZ} Hz")
plt.show()

The files carry no cell type. Baseline-epoch rates have a median of 1.05 Hz. 1351 of 1481 units
(91.2%) are below 5 Hz, the threshold used here for putative pyramidal cells; with 2.5 or 10 Hz the
count is 1205 or 1441. Spike times are sorted in every unit, and ISIs below 2 ms are rare (median
0.15% of a unit's ISIs, 99th percentile 2.6%).

In [ ]:
# The two sessions with pulse times: the pulse series, their stated units and values, and the
# position and laps.
opto = [s for s in sessions if s["pulses"]]
for s in opto:
    print(s["key"])
    for site, p in s["pulses"].items():
        d = (p["off"] - p["on"]) * 1000
        print(f"  {site}: {len(p['on'])} pulses, stored value(s) {p['values'].tolist()} '{p['unit']}', "
              f"duration median {np.median(d):.2f} ms (range {d.min():.2f} to {d.max():.2f}), "
              f"first {p['on'][0]:.1f} s, last {p['on'][-1]:.1f} s")
    print("  stimulation epochs:", [(e["behavioral_paradigm"], round(e["start"]), round(e["stop"]))
                                    for e in s["epochs"] if "uLED" in e["manipulation"]])
    x = s["pos_x"]
    print(f"  linearized position: {len(x)} samples, {np.nanmin(x):.1f} to {np.nanmax(x):.1f}, stated unit "
          f"{s['pos_attrs']['unit']!r} with conversion {s['pos_attrs']['conversion']}")
    laps = s["laps"]
    mz = v.epoch_interval(s, "Maze")[0]
    print(f"  laps: {len(laps)}, all inside the Maze epoch: {bool(np.all((laps[:, 0] >= mz[0]) & (laps[:, 1] <= mz[1])))}, "
          f"directions {np.unique(laps[:, 2], return_counts=True)[1].tolist()}, duration median {np.median(laps[:, 1] - laps[:, 0]):.1f} s")
    on = np.sort(np.concatenate([p["on"] for p in s["pulses"].values()]))
    per_lap = np.array([((on >= a) & (on < b)).sum() / (b - a) for a, b, _ in laps])
    no_stim = np.flatnonzero(per_lap < 1) + 1
    print(f"  laps with fewer than 1 pulse/s (numbered from 1): {no_stim.tolist()}")
    start_x = np.interp(laps[:, 0], s["pos_t"], s["pos_x"])
    stop_x = np.interp(laps[:, 1], s["pos_t"], s["pos_x"])
    for d in (0, 1):
        k = laps[:, 2] == d
        print(f"  direction {d}: position at lap start median {np.median(start_x[k]):.1f}, at lap end median {np.median(stop_x[k]):.1f}")

The pulse series are stated in watts but hold one value per uLED: 19 in sess9 and 62623 to 63599 in
sess12. These cannot be watts (the paper gives 0.02 to 0.1 uW), and with one value per uLED they
carry no intensity levels. Each pulse is four samples (off, on, on, off) spanning 19.87 ms (sess9)
or 19.63 ms (sess12), consistent with the paper's 20 ms pulses. The files hold series for six uLEDs
(Site3 to Site8) of the probe's twelve; Site6 and Site7 have only 120 to 221 pulses.

The linearized position runs from 0 to 110, the track's length in cm, but carries a conversion
factor of 100, which would make it 0 to 110 m; the conversion is wrong. Laps lie inside the Maze
epoch and alternate direction, starting and ending near the track ends. Three blocks of laps have no
pulses (sess9: laps 1-11, 52-61, 102-111; sess12: 1-10, 61-70, 121-132), the baseline runs of the
paper's design. In sess9 they are not at laps 61-70 and 121-130 as in Figure 3F's legend.

In [ ]:
# Clock checks. (1) Light responses: the population PSTH around the pulse onsets of each uLED,
# against the same onsets shifted by 250 ms. (2) Ripples: the population rate in the deposited
# ripple intervals against the same intervals shifted by 1 s, in every session with ripples.
for s in opto:
    pyr_spk = [x for x, p in zip(s["spikes"], s["pyr"]) if p]
    pre = v.epoch_interval(s, "PreStim")[0]
    for site, p in s["pulses"].items():
        on = p["on"][(p["on"] >= pre[0]) & (p["on"] < pre[1])]
        if len(on) < v.MIN_PULSES:
            continue
        true = np.mean([v.count_in_windows(x, on, 0.02).sum() for x in pyr_spk]) / len(on) / 0.02
        before = np.mean([v.count_in_windows(x, on - 0.02, 0.02).sum() for x in pyr_spk]) / len(on) / 0.02
        shifted = np.mean([v.count_in_windows(x, on + 0.25, 0.02).sum() for x in pyr_spk]) / len(on) / 0.02
        print(f"{s['key'].split('_ses-')[1]} {site}: mean pyramidal rate 0-20 ms after onset {true:.2f} Hz, "
              f"20 ms before {before:.2f} Hz, onsets shifted by 250 ms {shifted:.2f} Hz")
rip_rows = []
for s in sessions:
    r = s["ripples"]
    if len(r) < 50:
        continue
    allspk = np.sort(np.concatenate(s["spikes"]))
    dur = (r[:, 1] - r[:, 0]).sum()
    inside = (np.searchsorted(allspk, r[:, 1]) - np.searchsorted(allspk, r[:, 0])).sum() / dur
    sh = (np.searchsorted(allspk, r[:, 1] + 1) - np.searchsorted(allspk, r[:, 0] + 1)).sum() / dur
    rip_rows.append({"session": s["key"].split("_ses-")[1], "ripples": len(r),
                     "duration_ms_median": round(1000 * np.median(r[:, 1] - r[:, 0]), 1),
                     "peak_inside": bool(np.all((r[:, 2] >= r[:, 0]) & (r[:, 2] <= r[:, 1]))),
                     "pop_rate_in_ripples": round(inside, 1), "pop_rate_shifted_1s": round(sh, 1)})
rip_check = pd.DataFrame(rip_rows)
print(rip_check.to_string(index=False))
print("Sessions where the population rate inside ripples is at least twice the shifted rate:",
      int((rip_check.pop_rate_in_ripples > 2 * rip_check.pop_rate_shifted_1s).sum()), "of", len(rip_check))

Pulse onsets share the clock of the spikes: for every uLED tested, the mean rate of the putative
pyramidal cells in the 20 ms after onset is above the rate in the 20 ms before (by 0.10 to 1.25 Hz).
For 6 of 8 it is also above the rate in the same window 250 ms later. The other two are sess12
Site3 and Site5, and the later window often holds pulses of other uLEDs. Responses per unit are
tested under fig1e. The deposited ripples share the clock of the spikes: the population rate inside
them is at least twice the rate 1 s later in 22 of 25 sessions with ripples. In fCamk3 sess13
(1.3 times), fCamk3 sess17 (1.98 times) and fCamk5 sess10 (1.2 times; 217 events with a median
duration of 109 ms) the ripple intervals mark population bursts weakly. The LFP clock is checked in
the fig2b section: ripple-band power at the deposited ripple peaks is about 200 times its mean.
Position shares the clock of the laps (lap ends at track ends, above).

In [ ]:
# Responsiveness of every unit to every uLED, in the PreStim epoch (main) and over all
# stimulation epochs (reading b of fig1e). One generator per session.
for i, s in enumerate(opto):
    rng = np.random.default_rng(1000 + i)
    pre = v.epoch_interval(s, "PreStim")[0]
    s["resp_table"] = pd.DataFrame(v.responsive_sites(s, pre, rng))
    stim = [(e["start"], e["stop"]) for e in s["epochs"] if "uLED" in e["manipulation"]]
    whole = (min(a for a, _ in stim), max(b for _, b in stim))
    s["resp_table_all"] = pd.DataFrame(v.responsive_sites(s, whole, rng))
    t = s["resp_table"]
    s["resp_sites"] = {u: sorted(g[g.responsive.fillna(False).astype(bool)].site) for u, g in t.groupby("unit")}
    s["responsive"] = np.array([len(s["resp_sites"].get(u, [])) > 0 for u in range(len(s["spikes"]))])
    s["tested_sites"] = sorted(t[t.tested].site.unique())
    print(s["key"], "tested uLEDs:", s["tested_sites"], "; responsive units:", int(s["responsive"].sum()),
          "of", len(s["spikes"]), "; responsive putative pyramidal:", int((s["responsive"] & s["pyr"]).sum()),
          "of", int(s["pyr"].sum()))

## Claim cohort-counts

> CA1 pyramidal neurons were recorded and probed in four CaMKIIa-Cre::Ai32 mice, 822 pyramidal
> neurons in all, 43.3 plus or minus 8.37 per session.

In [ ]:
def cohort(rows):
    return {"sessions": len(rows), "mice": rows.subject.nunique(), "pyramidal": int(rows.putative_pyr.sum()),
            "per_session_mean": round(rows.putative_pyr.mean(), 1), "per_session_sd": round(rows.putative_pyr.std(), 2)}

readings = {"(a) all sessions": overview, "(b) sessions naming the uLED protocol": overview[overview.uLED_epochs]}
res = pd.DataFrame({k: cohort(r) for k, r in readings.items()}).T
res["mice_ok"] = res.mice == 4
res["total_ok"] = (res.pyramidal - 822).abs() <= 82
res["mean_ok"] = (res.per_session_mean - 43.3).abs() <= 8.37
print(res.to_string())
print("\nPer subject, reading (a):")
print(overview.groupby("subject").agg(sessions=("session", "size"), units=("units", "sum"),
                                       putative_pyr=("putative_pyr", "sum")).to_string())
print("\nVariant thresholds for putative pyramidal cells, reading (a) and (b):")
for thr in (2.5, 10):
    n = [int(sum((s["base_rate"] < thr).sum() for s in sessions)),
         int(sum((s["base_rate"] < thr).sum() for s in sessions if s["stim_protocol"]))]
    print(f"  below {thr} Hz: {n[0]} and {n[1]}")

**Result: partially reproduced.** Under reading (a), all 28 sessions, the dandiset holds 4 mice and
48.2 putative pyramidal cells per session (SD 12.18), within 43.3 plus or minus 8.37, but 1351 in
all, against 822 plus or minus 82. Under reading (b), the 22 sessions naming the uLED protocol, it
holds 3 mice (fCamk2 has no epoch labels), 1047 cells and 47.6 per session. At 43.3 per session,
822 cells would come from about 19 sessions; the dandiset holds 28 and does not say which the paper
used. The thresholds 2.5 and 10 Hz give 1205 and 1441 cells under reading (a).

## Claim fig1c-pulse-protocol

> uLEDs were driven with 20 ms pulses at random sites with randomly variable 20 to 40 ms offsets,
> so that the stimulation of each site recurred at about 0.3 to 0.6 s intervals.

In [ ]:
rows = []
for s in opto:
    stim = [(e["start"], e["stop"]) for e in s["epochs"] if "uLED" in e["manipulation"]]
    allon = []
    for site, p in s["pulses"].items():
        keep = np.zeros(len(p["on"]), bool)
        for a, b in stim:
            keep |= (p["on"] >= a) & (p["on"] < b)
        on, off = p["on"][keep], p["off"][keep]
        allon.append(np.column_stack([on, off]))
        ipi = np.diff(on)
        rows.append({"session": s["key"].split("_ses-")[1], "uLED": site, "pulses": len(on),
                     "duration_ms": round(1000 * np.median(off - on), 2),
                     "interval_median_s": round(np.median(ipi), 3),
                     "interval_5_95_s": np.percentile(ipi, [5, 95]).round(3).tolist(),
                     "intervals_in_0.3_0.6": round(np.mean((ipi >= 0.3) & (ipi <= 0.6)), 3)})
    iv = np.concatenate(allon)
    iv = iv[np.argsort(iv[:, 0])]
    gap = (iv[1:, 0] - iv[:-1, 1]) * 1000
    print(f"{s['key'].split('_ses-')[1]}: offset from the end of a pulse to the start of the next (any uLED), "
          f"median {np.median(gap):.1f} ms, 5th to 95th percentile {np.percentile(gap, 5):.1f} to {np.percentile(gap, 95):.1f} ms, "
          f"overlapping pulses {int((gap < 0).sum())} of {len(gap)}; within 20 to 40 ms {np.mean((gap >= 20) & (gap <= 40)):.1%}, "
          f"within 40 to 60 ms (file description) {np.mean((gap >= 40) & (gap <= 60)):.1%}; onset to next onset "
          f"within 40 to 60 ms {np.mean((np.diff(iv[:, 0]) >= 0.04) & (np.diff(iv[:, 0]) <= 0.06)):.1%}")
    s["gap_ms"] = gap
pp = pd.DataFrame(rows)
print(pp.to_string(index=False))
used = pp[pp.pulses >= v.MIN_PULSES]
print("\nMedian duration over uLEDs:", used.duration_ms.median(), "ms")
print("Median per-uLED interval of uLEDs with at least 1000 pulses, range:", used.interval_median_s.min(), "to", used.interval_median_s.max(), "s")
fig, axs = plt.subplots(1, 2, figsize=(9, 3))
for s in opto:
    g = s["gap_ms"]
    axs[0].hist(np.clip(g, 0, 400), bins=80, histtype="step", label=s["key"].split("_ses-")[1])
axs[0].set_xlabel("offset to next pulse, any uLED (ms; clipped at 400)")
axs[0].legend(fontsize=7)
s = opto[0]
p = s["pulses"]["Site3"]
axs[1].hist(np.clip(np.diff(p["on"]), 0, 3), bins=100, color="0.4")
axs[1].set_xlabel("interval between Site3 pulses, sess9 (s; clipped at 3)")
plt.tight_layout()
plt.show()
print("Values above the axes (clipped):", [int((s["gap_ms"] > 400).sum()) for s in opto], "offsets;",
      int((np.diff(p["on"]) > 3).sum()), "intervals")

**Result: partially reproduced.** Pulses last 19.75 ms (median over uLEDs), and the median interval
between pulses of one uLED is 0.281 to 0.383 s for the four uLEDs with at least 1000 pulses, both as
the paper states. The median offset from the end of one pulse to the start of the next is 60.4 ms
in sess9 and 80.0 ms in sess12, outside 18 to 44 ms. Of these offsets 31.8% and 30.2% fall within
20 to 40 ms, and the 95th percentiles are 241 and 328 ms. Measured onset to onset, the same 31.8%
and 30.2% fall within the 40 to 60 ms of the file's description, so the paper's range and the
file's agree if the file's is onset to onset. The longer gaps might come from the six uLEDs whose
series are not in the files; the outputs do not show this.

## Claim fig1e-responsive-fraction

> Of 822 pyramidal neurons, 611 responded to light from the uLEDs.

In [ ]:
rows = []
for s in opto:
    for label, t in (("(a) PreStim", s["resp_table"]), ("(b) all stimulation epochs", s["resp_table_all"])):
        resp = t[t.tested].groupby("unit").responsive.any().reindex(range(len(s["spikes"])), fill_value=False).values
        rows.append({"session": s["key"].split("_ses-")[1], "reading": label, "pyr": int(s["pyr"].sum()),
                     "pyr_responsive": int((resp & s["pyr"]).sum()), "all_units": len(resp), "all_responsive": int(resp.sum())})
fr = pd.DataFrame(rows)
print(fr.to_string(index=False))
tot = fr.groupby("reading")[["pyr", "pyr_responsive", "all_units", "all_responsive"]].sum()
tot["pct_pyr_responsive"] = (100 * tot.pyr_responsive / tot.pyr).round(1)
tot["within_tolerance"] = (tot.pct_pyr_responsive - 74.3).abs() <= 7.4
print(tot.to_string())
# Which part of the three-way test excludes cells: PreStim, putative pyramidal cells
parts = []
for s in opto:
    t = s["resp_table"][s["resp_table"].tested].copy()
    t = t[t.unit.isin(np.flatnonzero(s["pyr"]))]
    parts.append(t)
parts = pd.concat(parts)
print("\nPutative pyramidal unit-uLED tests in PreStim:", len(parts),
      "; above the bootstrap bound:", int((parts.rate_during > parts.boot_hi).sum()),
      "; z > 1.96:", int((parts.z > v.Z_THRESHOLD).sum()), "; KS p < 0.01:", int((parts.ks_p < v.KS_ALPHA).sum()),
      "; all three:", int(parts.responsive.sum()))
print("Variant, z > 1.96 alone (any uLED), PreStim:",
      sum(int(((s["resp_table"][s["resp_table"].tested].groupby("unit").z.max() > v.Z_THRESHOLD)
               .reindex(range(len(s["spikes"])), fill_value=False).values & s["pyr"]).sum()) for s in opto),
      "of", int(sum(s["pyr"].sum() for s in opto)))

**Result: reproduced** under reading (b). Over all stimulation epochs, 66 of 91 putative pyramidal
cells (72.5%) respond to at least one uLED, within 74.3 plus or minus 7.4%. Under reading (a), the
PreStim epoch alone, 57 of 91 (62.6%) do, outside tolerance. Of 364 unit and uLED tests in PreStim,
68 pass all three parts of the test. These are two sessions of one mouse, not the paper's 822 cells.

In [ ]:
# Light and control windows of each session, used by the claims below.
for s in opto:
    s["control"] = v.control_onsets(s)
    s["light_iv"] = v.all_light_intervals(s)
    s["cell_onsets"] = {u: np.sort(np.concatenate([s["pulses"][k]["on"] for k in sites]))
                        for u, sites in s["resp_sites"].items() if sites}
    s["all_tested_onsets"] = np.sort(np.concatenate([s["pulses"][k]["on"] for k in s["tested_sites"]]))
    print(s["key"].split("_ses-")[1], "control windows:", len(s["control"]),
          "; light-on time fraction over the stimulation epochs:",
          round((s["light_iv"][:, 1] - s["light_iv"][:, 0]).sum() /
                sum(e["stop"] - e["start"] for e in s["epochs"] if "uLED" in e["manipulation"]), 3))

## Claim fig1i-swr-decrease

> Light-induced spike responses of individual pyramidal cells decreased during sharp-wave ripples
> (n = 485 neurons, P < 1e-55, Wilcoxon paired signed-rank test).

In [ ]:
def overlaps(starts, width, iv):
    # whether [start, start + width) overlaps any interval of iv (sorted by start)
    j = np.searchsorted(iv[:, 0], starts + width) - 1
    ok = j >= 0
    out = np.zeros(len(starts), bool)
    out[ok] = iv[j[ok], 1] > starts[ok]
    return out

rows = []
for s in opto:
    home = [(e["start"], e["stop"]) for e in s["epochs"] if e["behavioral_paradigm"] in ("PreStim", "PostStim")]
    rip = s["ripples"][:, :2]
    ctrl = v.restrict(s["control"], home)
    c_in = overlaps(ctrl, v.CONTROL_WINDOW_S, rip)
    for u in np.flatnonzero(s["responsive"] & s["pyr"]):
        on = v.restrict(s["cell_onsets"][u], home)
        l_in = overlaps(on, v.PULSE_WINDOW_S, rip)
        x = s["spikes"][u]
        lc, cc = v.count_in_windows(x, on, 0.02), v.count_in_windows(x, ctrl, 0.02)
        rows.append({"session": s["key"].split("_ses-")[1], "unit": u,
                     "pulses_in": int(l_in.sum()), "pulses_out": int((~l_in).sum()),
                     "light_in": lc[l_in].mean() / 0.02, "light_out": lc[~l_in].mean() / 0.02,
                     "ctrl_in": cc[c_in].mean() / 0.02, "ctrl_out": cc[~c_in].mean() / 0.02,
                     "ctrl_windows_in": int(c_in.sum())})
swr = pd.DataFrame(rows)
swr["d_in"] = swr.light_in - swr.ctrl_in
swr["d_out"] = swr.light_out - swr.ctrl_out
swr["g_in"] = swr.light_in / swr.ctrl_in.replace(0, np.nan)
swr["g_out"] = swr.light_out / swr.ctrl_out.replace(0, np.nan)
print("Cells:", len(swr), "; pulses overlapping a ripple per cell, median", swr.pulses_in.median(),
      "(range", swr.pulses_in.min(), "to", swr.pulses_in.max(), "); control windows in ripples per session:",
      swr.groupby("session").ctrl_windows_in.first().to_dict())
print(swr[["light_in", "ctrl_in", "light_out", "ctrl_out", "d_in", "d_out"]].median().round(2).to_string())
for name, a, b in (("(a) difference light - control (Hz)", "d_in", "d_out"), ("(b) ratio light / control", "g_in", "g_out")):
    k = swr[[a, b]].dropna()
    k = k[np.isfinite(k).all(axis=1)]
    diff = k[a] - k[b]
    w = stats.wilcoxon(k[a], k[b])
    print(f"{name}: n = {len(k)}, median inside {k[a].median():.2f}, outside {k[b].median():.2f}, "
          f"median paired difference {diff.median():.2f}, lower inside in {(diff < 0).sum()} of {len(k)}, "
          f"Wilcoxon p = {w.pvalue:.2g}")
    for sess, g in k.assign(session=swr.loc[k.index, "session"]).groupby("session"):
        dd = g[a] - g[b]
        print(f"    {sess}: n = {len(g)}, median difference {dd.median():.2f}, lower inside {(dd < 0).sum()}, "
              f"p = {stats.wilcoxon(g[a], g[b]).pvalue:.2g}" if len(g) > 5 else f"    {sess}: n = {len(g)}")
fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(swr.d_out, swr.d_in, s=10, c="C0")
lim = [min(swr.d_out.min(), swr.d_in.min()), max(swr.d_out.max(), swr.d_in.max())]
ax.plot(lim, lim, "k:", lw=1)
ax.set_xlabel("light - control outside ripples (Hz)")
ax.set_ylabel("light - control inside ripples (Hz)")
plt.show()
print("Axes span every point: x", np.round([swr.d_out.min(), swr.d_out.max()], 1), "y", np.round([swr.d_in.min(), swr.d_in.max()], 1))

**Result: reproduced.** In 57 light-responsive putative pyramidal cells, the light minus control
rate is 0.30 Hz (median) in windows overlapping ripples and 2.68 Hz outside them. It is lower inside
ripples in 49 of 57 cells (Wilcoxon p = 6.8e-09), and in each session (p = 7.8e-06 and 9.5e-07).
As a ratio, light over control is 1.25 inside and 2.89 outside (53 of 56 lower, p = 8.6e-10). The
paper's n of 485 and P < 1e-55 are out of reach of two sessions.

## Claim figs10-swr-gain-vs-baseline-rate

> The firing-rate change of pyramidal neurons during SPW-Rs is negatively correlated with their
> baseline firing rate (rho = -0.19, P < 1e-7).

In [ ]:
rows = []
for s in sessions:
    e0 = s["epochs"][0]
    r = s["ripples"]
    r = r[(r[:, 0] >= e0["start"]) & (r[:, 1] < e0["stop"])]
    if len(r) < 50:
        continue
    t_in = (r[:, 1] - r[:, 0]).sum()
    t_out = (e0["stop"] - e0["start"]) - t_in
    for u in np.flatnonzero(s["pyr"]):
        x = s["spikes"][u]
        n_all = ((x >= e0["start"]) & (x < e0["stop"])).sum()
        n_in = (np.searchsorted(x, r[:, 1]) - np.searchsorted(x, r[:, 0])).sum()
        rows.append({"session": s["key"].split("_ses-")[1], "subject": s["subject"], "base": n_all / (e0["stop"] - e0["start"]),
                     "in": n_in / t_in, "out": (n_all - n_in) / t_out, "ripples": len(r)})
g = pd.DataFrame(rows)
g["log2_ratio"] = np.log2(g["in"] / g["out"])
g["diff"] = g["in"] - g["out"]
print("Sessions:", g.session.nunique(), "; putative pyramidal cells:", len(g),
      "; cells with no spikes in ripples (left out of reading a):", int((g["in"] == 0).sum()))
for name, col in (("(a) log2 ratio in/out", "log2_ratio"), ("(b) difference in - out", "diff")):
    k = g[np.isfinite(g[col])]
    rho = stats.spearmanr(k[col], k.base)
    print(f"{name}: n = {len(k)}, rho = {rho.statistic:.3f}, p = {rho.pvalue:.2g}, "
          f"within -0.19 +- 0.068: {abs(rho.statistic + 0.19) <= 0.068}")
print("\nPer subject and per session, reading (a):")
k = g[np.isfinite(g.log2_ratio)]
for key, grp in list(k.groupby("subject")) + list(k.groupby("session")):
    rr = stats.spearmanr(grp.log2_ratio, grp.base)
    print(f"  {key}: n = {len(grp)}, rho = {rr.statistic:.2f}, p = {rr.pvalue:.2g}")
bad = rip_check[rip_check.pop_rate_in_ripples < 2 * rip_check.pop_rate_shifted_1s].session.tolist()
kk = k[~k.session.isin(bad)]
rr = stats.spearmanr(kk.log2_ratio, kk.base)
print(f"Variant, without the sessions whose ripples fail the clock check {bad}: n = {len(kk)}, rho = {rr.statistic:.3f}, p = {rr.pvalue:.2g}")
kk = g[g.ripples >= 1000]
kk = kk[np.isfinite(kk.log2_ratio)]
rr = stats.spearmanr(kk.log2_ratio, kk.base)
print(f"Variant, sessions with at least 1000 ripples in the first epoch: n = {len(kk)}, rho = {rr.statistic:.3f}, p = {rr.pvalue:.2g}")
print("Variant, base rate measured outside ripples (not the same spikes as the change), reading (a):",
      stats.spearmanr(k.log2_ratio, k.out).statistic.round(3))
fig, ax = plt.subplots(figsize=(4, 3.5))
ax.scatter(np.log10(k.base), k.log2_ratio, s=4, alpha=0.5)
ax.set_xlabel("log10 baseline rate (Hz)")
ax.set_ylabel("log2 rate in / out of ripples")
plt.show()
print("Range plotted: x", np.round([np.log10(k.base).min(), np.log10(k.base).max()], 2), "y", np.round([k.log2_ratio.min(), k.log2_ratio.max()], 2))

**Result: partially reproduced.** Over 1166 putative pyramidal cells of 24 sessions, the log2 ratio
of the rate inside ripples to the rate outside is negatively correlated with the baseline rate,
rho = -0.437 (p = 1.8e-55). It is negative in every subject (-0.37 to -0.71) and every session, and
stronger than the paper's -0.19 plus or minus 0.068. Under the other reading, the difference inside
minus outside, rho = -0.002 (p = 0.95). Without the three sessions whose ripples fail the clock
check, rho = -0.468. Which measure of change the paper used is not known, and the two readings
disagree.

## Claim fig2b-theta-trough-gain

> During track running, the light-induced rate gain is largest at the trough of the theta cycle
> (P < 1e-6, Wilcoxon test).

In [ ]:
rows, check = [], []
for s in opto:
    b0 = v.epoch_interval(s, "BaselinePre")[0]
    t0 = b0[0] + 600
    lfp = v.read_lfp(s["full_url"], t0, t0 + 300)
    peaks = s["ripples"][:, 2]
    peaks = peaks[(peaks > t0) & (peaks < t0 + 300)]
    ch = v.ripple_channel(lfp, t0, peaks)
    mz = v.epoch_interval(s, "Maze")[0]
    x = v.read_lfp(s["full_url"], mz[0], mz[1], [ch])[:, 0]
    phase = v.theta_phase(x)
    # clock check of the LFP: ripple-band power at ripple peaks against shifted peaks
    from scipy import signal as sg
    sos = sg.butter(3, v.RIPPLE_BAND, btype="band", fs=v.LFP_RATE, output="sos")
    y = sg.sosfiltfilt(sos, lfp[:, ch].astype(float)) ** 2
    ii = ((peaks - t0) * v.LFP_RATE).astype(int)
    ii = ii[(ii > 2000) & (ii < len(y) - 2000)]
    check.append((s["key"].split("_ses-")[1], ch, len(ii), y[ii].mean() / y.mean(), y[ii + 1250].mean() / y.mean()))
    def ph(t):
        i = ((t - mz[0]) * v.LFP_RATE).astype(int)
        return phase[np.clip(i, 0, len(phase) - 1)]
    laps = s["laps"][:, :2]
    ctrl = s["control"][v.in_intervals(s["control"], laps)]
    c_ph = ph(ctrl + 0.01)
    for u in np.flatnonzero(s["responsive"] & s["pyr"]):
        on = s["cell_onsets"][u]
        on = on[v.in_intervals(on, laps)]
        l_ph = ph(on + 0.01)
        xs = s["spikes"][u]
        lc, cc = v.count_in_windows(xs, on, 0.02) / 0.02, v.count_in_windows(xs, ctrl, 0.02) / 0.02
        row = {"session": s["key"].split("_ses-")[1], "unit": u}
        trough_l, trough_c = np.abs(l_ph) > np.pi / 2, np.abs(c_ph) > np.pi / 2
        row["gain_trough"] = lc[trough_l].mean() - cc[trough_c].mean()
        row["gain_peak"] = lc[~trough_l].mean() - cc[~trough_c].mean()
        edges = np.linspace(-np.pi, np.pi, 7)
        for k in range(6):
            a = (l_ph >= edges[k]) & (l_ph < edges[k + 1])
            b = (c_ph >= edges[k]) & (c_ph < edges[k + 1])
            row[f"bin{k}"] = lc[a].mean() - cc[b].mean()
        rows.append(row)
for c in check:
    print(f"{c[0]}: LFP channel {c[1]}; ripple-band power at {c[2]} ripple peaks {c[3]:.1f} x the mean, "
          f"1 s after them {c[4]:.1f} x")
th = pd.DataFrame(rows)
d = th.gain_trough - th.gain_peak
w = stats.wilcoxon(th.gain_trough, th.gain_peak)
print(f"\nCells: {len(th)}; gain trough half median {th.gain_trough.median():.2f} Hz, peak half {th.gain_peak.median():.2f} Hz; "
      f"trough higher in {(d > 0).sum()} of {len(d)}; median difference {d.median():.2f} Hz; Wilcoxon p = {w.pvalue:.2g}")
for sess, grp in th.groupby("session"):
    dd = grp.gain_trough - grp.gain_peak
    print(f"  {sess}: n = {len(grp)}, median difference {dd.median():.2f}, p = {stats.wilcoxon(grp.gain_trough, grp.gain_peak).pvalue:.2g}")
centres = np.degrees((np.linspace(-np.pi, np.pi, 7)[:-1] + np.pi / 6)).round()
binmed = th[[f"bin{k}" for k in range(6)]].median()
print("Variant, median gain (Hz) in 60-degree bins centred at", centres.tolist(), "(0 = peak, +-180 = trough):",
      binmed.round(2).tolist(), "; largest at", centres[int(np.argmax(binmed.values))])

**Result: not reproduced.** In 57 light-responsive putative pyramidal cells during laps, the gain
(light minus control) is 2.04 Hz in the trough half of theta and 2.07 Hz in the peak half (medians).
The trough is higher in 22 of 57 cells (Wilcoxon p = 0.14), and in neither session (p = 0.46 and
0.16). In 60-degree bins the median gain is largest at 30 degrees past the peak (2.13 Hz), and
1.88 and 1.92 Hz in the bins next to the trough. The LFP channel, band and phase convention are this
attempt's choices, since the Methods were not available.

In [ ]:
# Rate maps and place fields over the laps of the Maze epoch, per running direction.
for s in opto:
    grid = v.lap_grid(s, s["laps"])
    s["grid"] = grid
    on_any = v.in_intervals(grid["t"], s["light_iv"])
    s["ctrl_mask"] = ~on_any
    s["ctrl_maps"] = [v.rate_map(x, grid, s["ctrl_mask"]) for x in s["spikes"]]
    s["ctrl_fields"] = [[v.find_fields(m[d]) for d in (0, 1)] for m in s["ctrl_maps"]]
    s["place"] = np.array([any(len(f) for f in ff) for ff in s["ctrl_fields"]])
    # light masks: the cell's own uLEDs, and every tested uLED
    def mask_for(onsets):
        iv = np.column_stack([onsets, onsets + v.PULSE_WINDOW_S])
        return v.in_intervals(grid["t"], v.merge_intervals(iv))
    s["light_mask"] = {u: mask_for(on) for u, on in s["cell_onsets"].items()}
    s["all_light_mask"] = mask_for(s["all_tested_onsets"])
    s["light_maps"] = {u: v.rate_map(s["spikes"][u], grid, m) for u, m in s["light_mask"].items()}
    s["all_light_maps"] = [v.rate_map(x, grid, s["all_light_mask"]) for x in s["spikes"]]
    print(f"{s['key'].split('_ses-')[1]}: grid {len(grid['t'])} steps of {v.GRID_S * 1000:.0f} ms over {len(s['laps'])} laps; "
          f"time without light {s['ctrl_mask'].mean():.1%}; time in the light windows of all tested uLEDs {s['all_light_mask'].mean():.1%}; "
          f"place cells among putative pyramidal {int((s['place'] & s['pyr']).sum())} of {int(s['pyr'].sum())}")

## Claim figs11-place-cell-fraction

> More than half of the pyramidal neurons were place cells, 73% of light-responsive and 71% of
> non-responsive neurons, with no difference between them (P = 0.90, chi-squared test).

In [ ]:
rows = []
for s in opto:
    for u in np.flatnonzero(s["pyr"]):
        rows.append({"session": s["key"].split("_ses-")[1], "unit": u, "responsive": bool(s["responsive"][u]),
                     "place": bool(s["place"][u]), "n_fields": sum(len(f) for f in s["ctrl_fields"][u])})
pc = pd.DataFrame(rows)
tab = pd.crosstab(pc.responsive, pc.place)
print(tab)
frac = pc.groupby("responsive").place.mean() * 100
print("Place cells among responsive: %.1f%%, among non-responsive: %.1f%%" % (frac[True], frac[False]))
chi = stats.chi2_contingency(tab.values)
print(f"Chi-squared (Yates) = {chi.statistic:.3f}, p = {chi.pvalue:.3f}; without correction p = {stats.chi2_contingency(tab.values, correction=False).pvalue:.3f}")
print("Per session:")
print(pc.groupby(["session", "responsive"]).place.agg(["mean", "size"]).round(3).to_string())
print("Place cells over all putative pyramidal cells: %.1f%%; fields per place cell, median %d" %
      (100 * pc.place.mean(), pc[pc.place].n_fields.median()))
# Threshold variant: minimum peak rate of a field
for peak in (1.0, 3.0):
    old = v.FIELD_MIN_PEAK_HZ
    v.FIELD_MIN_PEAK_HZ = peak
    pl = np.concatenate([[any(len(v.find_fields(s["ctrl_maps"][u][d])) for d in (0, 1)) for u in np.flatnonzero(s["pyr"])] for s in opto])
    v.FIELD_MIN_PEAK_HZ = old
    resp = pc.responsive.values
    print(f"Variant, field peak at least {peak} Hz: place cells {100 * pl[resp].mean():.1f}% of responsive, {100 * pl[~resp].mean():.1f}% of non-responsive")

**Result: not reproduced.** 45 of 57 light-responsive putative pyramidal cells (78.9%) and 18 of 34
non-responsive ones (52.9%) are place cells. The paper's chi-squared test finds a difference
(p = 0.018), where the paper found none (P = 0.90). Both sessions go the same way (71% against 44%,
and 91% against 61%), and so do field peak thresholds of 1 and 3 Hz (86.0% against 67.6%, and 68.4%
against 44.1%). The responsive fraction is within tolerance of 73%; the non-responsive one is not
within 71%. Over all putative pyramidal cells, 69.2% are place cells, more than half as stated.

## Claim fig2e-infield-gain

> The light-induced rate increase is higher within than outside the place field (n = 553 place
> fields, P < 1e-115, Wilcoxon test).

In [ ]:
def rate_where(x, grid, mask):
    # rate (Hz) of spikes falling in grid steps where mask holds
    i = np.searchsorted(grid["t"], x, side="right") - 1
    ok = (i >= 0) & (i < len(grid["t"]))
    i = i[ok][(x[ok] - grid["t"][i[ok]]) < v.GRID_S]
    t = mask.sum() * v.GRID_S
    return mask[i].sum() / t if t > 0 else np.nan, t

rows = []
for s in opto:
    grid = s["grid"]
    b = v.bin_index(grid["x"])
    for u in np.flatnonzero(s["responsive"] & s["pyr"] & s["place"]):
        for d in (0, 1):
            for (a, z, peak) in s["ctrl_fields"][u][d]:
                infield = (grid["dir"] == d) & (b >= a) & (b < z)
                outfield = (grid["dir"] == d) & ~((b >= a) & (b < z))
                li, lt = rate_where(s["spikes"][u], grid, s["light_mask"][u] & infield)
                ci, _ = rate_where(s["spikes"][u], grid, s["ctrl_mask"] & infield)
                lo, _ = rate_where(s["spikes"][u], grid, s["light_mask"][u] & outfield)
                co, _ = rate_where(s["spikes"][u], grid, s["ctrl_mask"] & outfield)
                rows.append({"session": s["key"].split("_ses-")[1], "unit": u, "dir": d, "start": a, "stop": z,
                             "light_in": li, "ctrl_in": ci, "light_out": lo, "ctrl_out": co, "light_s_in_field": lt,
                             "home_rate": s["base_rate"][u]})
inf = pd.DataFrame(rows)
inf["gain_in"] = inf.light_in - inf.ctrl_in
inf["gain_out"] = inf.light_out - inf.ctrl_out
print("Place fields of responsive putative pyramidal cells:", len(inf), "from", inf.groupby(["session", "unit"]).ngroups,
      "cells; light time in field per field, median", round(inf.light_s_in_field.median(), 1), "s")
print(inf[["light_in", "ctrl_in", "light_out", "ctrl_out", "gain_in", "gain_out"]].median().round(2).to_string())
d = inf.gain_in - inf.gain_out
w = stats.wilcoxon(inf.gain_in, inf.gain_out)
print(f"In-field gain higher in {(d > 0).sum()} of {len(d)}; median difference {d.median():.2f} Hz; Wilcoxon p = {w.pvalue:.2g}")
for sess, grp in inf.groupby("session"):
    dd = grp.gain_in - grp.gain_out
    print(f"  {sess}: n = {len(grp)}, median difference {dd.median():.2f}, p = {stats.wilcoxon(grp.gain_in, grp.gain_out).pvalue:.2g}")
ratio_in = inf.light_in / inf.ctrl_in.replace(0, np.nan)
ratio_out = inf.light_out / inf.ctrl_out.replace(0, np.nan)
k = np.isfinite(ratio_in) & np.isfinite(ratio_out)
print(f"Variant, ratio light/control: in field median {ratio_in[k].median():.2f}, out {ratio_out[k].median():.2f}, "
      f"n = {k.sum()}, p = {stats.wilcoxon(ratio_in[k], ratio_out[k]).pvalue:.2g}")

**Result: reproduced.** Over 64 control place fields of 45 light-responsive putative pyramidal cells,
the gain is 4.23 Hz in the field and 1.01 Hz outside it (medians). In-field gain is higher for 51 of
64 fields (Wilcoxon p = 9.1e-08), and in both sessions (p = 3.8e-07 and 0.0032). As a ratio of light
to control rate, the in-field ratio is lower (2.16 against 2.89, p = 0.00038), so the increase is
larger in hertz but not relative to the control rate.

## Claim figs12-gain-rate-correlations

> The in-field gain is positively correlated with the out-of-field firing rate (rho = 0.17,
> P < 1e-5) and with the home-cage firing rate of the neuron (rho = 0.25, P < 1e-7).

In [ ]:
r1 = stats.spearmanr(inf.gain_in, inf.ctrl_out)
r2 = stats.spearmanr(inf.gain_in, inf.home_rate)
print(f"n = {len(inf)} fields. In-field gain against out-of-field rate: rho = {r1.statistic:.3f}, p = {r1.pvalue:.2g}, "
      f"within 0.17 +- 0.084: {abs(r1.statistic - 0.17) <= 0.084}")
print(f"In-field gain against home-cage rate: rho = {r2.statistic:.3f}, p = {r2.pvalue:.2g}, "
      f"within 0.25 +- 0.084: {abs(r2.statistic - 0.25) <= 0.084}")
for sess, grp in inf.groupby("session"):
    print(f"  {sess}: n = {len(grp)}, rho out-of-field {stats.spearmanr(grp.gain_in, grp.ctrl_out).statistic:.2f}, "
          f"rho home cage {stats.spearmanr(grp.gain_in, grp.home_rate).statistic:.2f}")
one = inf.sort_values("gain_in", ascending=False).groupby(["session", "unit"]).first()
print(f"Variant, one field per cell (n = {len(one)}): rho out-of-field {stats.spearmanr(one.gain_in, one.ctrl_out).statistic:.3f}, "
      f"rho home cage {stats.spearmanr(one.gain_in, one.home_rate).statistic:.3f}")

**Result: not reproduced.** Over 64 fields, the in-field gain is correlated with neither the
out-of-field rate (rho = 0.051, p = 0.69) nor the home-cage rate (rho = -0.065, p = 0.61); both
are outside tolerance of 0.17 and 0.25. With one field per cell (45 cells), rho is 0.207 for the
out-of-field rate, within tolerance, and -0.005 for the home-cage rate.

## Claim fig3a-place-cells-more-excitable

> Light responses in the home cage before the track were stronger in place cells than in
> non-place cells (P < 1e-4, Kruskal-Wallis test).

In [ ]:
rows = []
for s in opto:
    t = s["resp_table"][s["resp_table"].tested].copy()
    t["resp_hz"] = t.rate_during - t.rate_before
    best = t.groupby("unit").agg(resp_hz=("resp_hz", "max"), z=("z", "max"))
    for u in np.flatnonzero(s["pyr"]):
        rows.append({"session": s["key"].split("_ses-")[1], "unit": u, "place": bool(s["place"][u]),
                     "resp_hz": best.resp_hz.get(u, np.nan), "z": best.z.get(u, np.nan)})
ex = pd.DataFrame(rows).dropna()
for col in ("resp_hz", "z"):
    a, b = ex[ex.place][col], ex[~ex.place][col]
    kw = stats.kruskal(a, b)
    print(f"{col}: place cells n = {len(a)}, median {a.median():.2f}; non-place n = {len(b)}, median {b.median():.2f}; "
          f"Kruskal-Wallis p = {kw.pvalue:.2g}")
for sess, grp in ex.groupby("session"):
    a, b = grp[grp.place].resp_hz, grp[~grp.place].resp_hz
    print(f"  {sess}: place {len(a)} median {a.median():.2f}, non-place {len(b)} median {b.median():.2f}, p = {stats.kruskal(a, b).pvalue:.2g}")
# firing rate as a confound: the comparison within baseline-rate halves
ex["base"] = [s["base_rate"][u] for sess, u in zip(ex.session, ex.unit) for s in opto if s["key"].endswith(sess)]
med = ex.base.median()
for name, part in (("below", ex[ex.base < med]), ("above", ex[ex.base >= med])):
    a, b = part[part.place].resp_hz, part[~part.place].resp_hz
    print(f"  baseline rate {name} the median: place {len(a)} median {a.median():.2f}, non-place {len(b)} median {b.median():.2f}, "
          f"p = {stats.kruskal(a, b).pvalue:.2g}")

**Result: reproduced.** In the home cage (PreStim), the light response (rate during the pulse minus
before it, best uLED) has a median of 2.07 Hz in 63 place cells and 0.94 Hz in 28 non-place cells
(Kruskal-Wallis p = 0.0041; z-scored, p = 0.0021), and the same holds in both sessions (p = 0.01 and
0.029). The paper says the difference is not explained by firing rate. Here, among cells below the
median baseline rate it is small and not significant (1.83 against 1.33 Hz, p = 0.32); among the
faster half it is 3.08 against 0.58 Hz (p = 0.0021).

## Claim fig3c-unmasked-fields

> Optogenetic depolarization revealed place fields in the majority of non-place cells, 289 of
> 417 (69.3%).

In [ ]:
rows = []
for s in opto:
    for u in np.flatnonzero(s["pyr"] & ~s["place"]):
        r = {"session": s["key"].split("_ses-")[1], "unit": u, "responsive": bool(s["responsive"][u])}
        if r["responsive"]:
            r["light_fields_a"] = [v.find_fields(s["light_maps"][u][d]) for d in (0, 1)]
        r["light_fields_b"] = [v.find_fields(s["all_light_maps"][u][d]) for d in (0, 1)]
        rows.append(r)
nonp = pd.DataFrame(rows)
nonp["unmasked_a"] = nonp.apply(lambda r: any(len(f) for f in r.light_fields_a) if r.responsive else False, axis=1)
nonp["unmasked_b"] = nonp.light_fields_b.apply(lambda ff: any(len(f) for f in ff))
ra = nonp[nonp.responsive]
pa = 100 * ra.unmasked_a.mean()
pb = 100 * nonp.unmasked_b.mean()
print(f"(a) light-responsive non-place cells: {int(ra.unmasked_a.sum())} of {len(ra)} with a light field ({pa:.1f}%), "
      f"within 69.3 +- 6.9: {abs(pa - 69.3) <= 6.9}")
print(f"(b) all non-place cells, every tested uLED: {int(nonp.unmasked_b.sum())} of {len(nonp)} ({pb:.1f}%), "
      f"within 69.3 +- 6.9: {abs(pb - 69.3) <= 6.9}")
print(nonp.groupby(["session", "responsive"])[["unmasked_a", "unmasked_b"]].agg(["sum", "size"]).to_string())
# Null: the same field rule on control maps made from as little time as the light maps
rng = np.random.default_rng(31)
null = []
for s in opto:
    frac = s["all_light_mask"].mean() / s["ctrl_mask"].mean()
    for u in np.flatnonzero(s["pyr"] & ~s["place"]):
        keep = s["ctrl_mask"] & (rng.random(len(s["ctrl_mask"])) < frac)
        m = v.rate_map(s["spikes"][u], s["grid"], keep)
        null.append(any(len(v.find_fields(m[d])) for d in (0, 1)))
print(f"Reference, non-place cells with a field in a control map thinned at random to the time of the light maps: "
      f"{sum(null)} of {len(null)} ({100 * np.mean(null):.1f}%)")

**Result: not reproduced.** Under reading (a), 10 of 12 light-responsive non-place cells (83.3%) have
a light field; under reading (b), 12 of 28 non-place cells (42.9%). The paper's value is 69.3 plus
or minus 6.9%. Reading (a) rests on 12 cells. Reading (b) is close to the 35.7% (10 of 28) that the
same field rule finds in control maps thinned at random to the time of the light maps. The two
sessions hold 28 non-place cells, against the paper's 417.

In [ ]:
# Field groups for figures 3E to 3G: control fields of light-responsive place cells, and light
# fields (reading a) of light-responsive non-place cells, each with its direction.
groups = []
for s in opto:
    occ_ctrl = {d: np.bincount(v.bin_index(s["grid"]["x"][(s["grid"]["dir"] == d) & s["ctrl_mask"]]), minlength=v.N_BINS) for d in (0, 1)}
    for u in np.flatnonzero(s["responsive"] & s["pyr"]):
        lm, cm = s["light_maps"][u], s["ctrl_maps"][u]
        occ_l = {d: np.bincount(v.bin_index(s["grid"]["x"][(s["grid"]["dir"] == d) & s["light_mask"][u]]), minlength=v.N_BINS) for d in (0, 1)}
        fields = s["ctrl_fields"][u] if s["place"][u] else [v.find_fields(lm[d]) for d in (0, 1)]
        for d in (0, 1):
            if not fields[d]:
                continue
            groups.append({"session": s["key"].split("_ses-")[1], "unit": u, "dir": d,
                           "group": "place" if s["place"][u] else "unmasked",
                           "peak_ctrl": int(np.nanargmax(cm[d])) if np.any(cm[d] > 0) else np.nan,
                           "peak_light": int(np.nanargmax(lm[d])) if np.any(lm[d] > 0) else np.nan,
                           "si_ctrl": v.spatial_info(cm[d], occ_ctrl[d]), "si_light": v.spatial_info(lm[d], occ_l[d])})
fg = pd.DataFrame(groups)
print(fg.groupby("group").size().to_string())
# Variant control: per cell, a random draw of control windows (20 ms before a pulse, no uLED on)
# as many as the cell's light windows over the laps, as the main text's "random intervals (20 ms)
# between the light pulses" may mean. Used as a variant in figures 3F and 3G.
rng = np.random.default_rng(30)
for s in opto:
    laps = s["laps"][:, :2]
    ctrl = s["control"][v.in_intervals(s["control"], laps)]
    s["ctrl_matched_mask"] = {}
    for u, on in s["cell_onsets"].items():
        n = int(v.in_intervals(on, laps).sum())
        pick = np.sort(rng.choice(ctrl, size=min(n, len(ctrl)), replace=False))
        iv = v.merge_intervals(np.column_stack([pick, pick + v.CONTROL_WINDOW_S]))
        s["ctrl_matched_mask"][u] = v.in_intervals(s["grid"]["t"], iv)
print("Matched control time as a fraction of light time, median over cells:",
      round(np.median([s["ctrl_matched_mask"][u].sum() / s["light_mask"][u].sum() for s in opto for u in s["ctrl_matched_mask"]]), 3))

## Claim fig3e-ghost-fields

> The spatial location of light-induced spikes matches that of spikes without stimulation, both
> for place fields of place cells (P < 1e-40) and for unmasked place fields of non-place cells,
> whose sparse control spikes form ghost fields (P < 1e-10).

In [ ]:
rng = np.random.default_rng(32)
for grp, k in fg.dropna(subset=["peak_ctrl", "peak_light"]).groupby("group"):
    r = stats.pearsonr(k.peak_ctrl, k.peak_light)
    same = np.mean(np.abs(k.peak_ctrl - k.peak_light) <= 2)
    sh = np.array([np.mean(np.abs(k.peak_ctrl.values - rng.permutation(k.peak_light.values)) <= 2) for _ in range(500)])
    print(f"{grp}: n = {len(k)} field directions, Pearson r = {r.statistic:.3f}, p = {r.pvalue:.2g}; "
          f"peaks within 2 bins (4.4 cm): {same:.1%}, shuffled pairings {sh.mean():.1%} (95th percentile {np.percentile(sh, 95):.1%}), "
          f"shuffle p = {(np.sum(sh >= same) + 1) / 501:.3f}")
    for sess, kk in k.groupby("session"):
        if len(kk) > 3:
            print(f"    {sess}: n = {len(kk)}, r = {stats.pearsonr(kk.peak_ctrl, kk.peak_light).statistic:.2f}")
print("Field directions dropped for an empty map:", int(fg[["peak_ctrl", "peak_light"]].isna().any(axis=1).sum()))
fig, axs = plt.subplots(1, 2, figsize=(7, 3.3))
for ax, (grp, k) in zip(axs, fg.dropna(subset=["peak_ctrl", "peak_light"]).groupby("group")):
    ax.scatter(k.peak_ctrl * v.TRACK_CM / v.N_BINS, k.peak_light * v.TRACK_CM / v.N_BINS, s=10)
    ax.set_title(grp)
    ax.set_xlabel("control peak (cm)")
    ax.set_ylabel("light peak (cm)")
    ax.set_xlim(0, 110)
    ax.set_ylim(0, 110)
plt.tight_layout()
plt.show()

**Result: partially reproduced.** For 61 control fields of light-responsive place cells, the light
map's peak follows the control map's peak (Pearson r = 0.857, p = 1.3e-18). For 19 light fields of
non-place cells, r = 0.310 (p = 0.2), so the criterion's test fails for this group. In the shuffle
variant, 57.9% of these peaks are within 4.4 cm of the control peak, against 14.0% for shuffled
pairings (p = 0.002). Of the 19 unmasked fields, 15 come from sess9.

## Claim fig3f-light-stability

> Place fields during light stimulation were more stable between the first and second halves of
> the session than during control epochs, for both true place fields and unmasked place fields
> (P < 1e-9 and P < 1e-19, two-way ANOVA).

In [ ]:
rows = []
for s in opto:
    grid = s["grid"]
    lap_no = grid["lap"] + 1
    h1 = (lap_no >= 11) & (lap_no <= 60)
    h2 = (lap_no >= 71) & (lap_no <= 120)
    sub = fg[fg.session == s["key"].split("_ses-")[1]]
    for _, r in sub.iterrows():
        u, d = r.unit, r.dir
        lm1 = v.rate_map(s["spikes"][u], grid, s["light_mask"][u] & h1)[d]
        lm2 = v.rate_map(s["spikes"][u], grid, s["light_mask"][u] & h2)[d]
        cm1 = v.rate_map(s["spikes"][u], grid, s["ctrl_mask"] & h1)[d]
        cm2 = v.rate_map(s["spikes"][u], grid, s["ctrl_mask"] & h2)[d]
        mm1 = v.rate_map(s["spikes"][u], grid, s["ctrl_matched_mask"][u] & h1)[d]
        mm2 = v.rate_map(s["spikes"][u], grid, s["ctrl_matched_mask"][u] & h2)[d]
        rows.append({"session": r.session, "unit": u, "dir": d, "group": r.group,
                     "stab_light": v.map_corr(lm1, lm2), "stab_ctrl": v.map_corr(cm1, cm2),
                     "stab_ctrl_matched": v.map_corr(mm1, mm2)})
st = pd.DataFrame(rows)
print("Laps per session:", {s["key"].split("_ses-")[1]: len(s["laps"]) for s in opto}, "; halves are laps 11-60 and 71-120")
for grp, k in st.groupby("group"):
    k2 = k.dropna()
    w = stats.wilcoxon(k2.stab_light, k2.stab_ctrl)
    dd = k2.stab_light - k2.stab_ctrl
    print(f"{grp}: n = {len(k2)} (dropped {len(k) - len(k2)} with a flat or empty half map); stability light median "
          f"{k2.stab_light.median():.3f}, control {k2.stab_ctrl.median():.3f}; light higher in {(dd > 0).sum()}; "
          f"Wilcoxon p = {w.pvalue:.2g}")
for grp, k in st.groupby("group"):
    k2 = k.dropna(subset=["stab_light", "stab_ctrl_matched"])
    dd = k2.stab_light - k2.stab_ctrl_matched
    print(f"Variant, control from time-matched control windows, {grp}: n = {len(k2)}, light median {k2.stab_light.median():.3f}, "
          f"matched control {k2.stab_ctrl_matched.median():.3f}; light higher in {(dd > 0).sum()}; Wilcoxon p = "
          f"{stats.wilcoxon(k2.stab_light, k2.stab_ctrl_matched).pvalue:.2g}")
import statsmodels.formula.api as smf
import statsmodels.api as sm
long = pd.concat([st.assign(cond="light", stab=st.stab_light), st.assign(cond="control", stab=st.stab_ctrl)]).dropna(subset=["stab"])
an = sm.stats.anova_lm(smf.ols("stab ~ C(cond) * C(group)", data=long).fit(), typ=2)
print("\nVariant, two-way ANOVA of stability on condition and group:")
print(an.round(4).to_string())

**Result: not reproduced.** Between laps 11-60 and 71-120, light maps are less stable than control
maps. The medians are 0.675 against 0.797 for place fields (light higher in 15 of 61, p = 6.2e-05)
and 0.278 against 0.610 for unmasked fields (light higher in 7 of 19, p = 0.087). Control maps use
88% to 90% of lap time; the light maps of all tested uLEDs use 11% to 12%. Against control maps from
as many 20 ms control windows as light windows, the differences shrink and are not significant
(0.675 against 0.732, p = 0.95; 0.278 against 0.449, p = 0.71), and light is still not more stable.

## Claim fig3g-spatial-information

> Spatial information of control and light-stimulated epochs is correlated, and the light-boosted
> increase is stronger for unmasked place fields of non-place cells than for place cells
> (P < 1e-15, Kruskal-Wallis test).

In [ ]:
k = fg.dropna(subset=["si_ctrl", "si_light"]).copy()
k["boost"] = k.si_light - k.si_ctrl
for grp, kk in k.groupby("group"):
    r = stats.spearmanr(kk.si_ctrl, kk.si_light)
    print(f"{grp}: n = {len(kk)}, SI control median {kk.si_ctrl.median():.2f}, light {kk.si_light.median():.2f} bits/spike; "
          f"Spearman rho = {r.statistic:.3f}, p = {r.pvalue:.2g}; increase median {kk.boost.median():.2f}")
kw = stats.kruskal(k[k.group == "unmasked"].boost, k[k.group == "place"].boost)
print(f"Increase, unmasked against place: Kruskal-Wallis p = {kw.pvalue:.2g}")
print("Field directions dropped for an empty map:", len(fg) - len(k))
si_m = []
for _, r in k.iterrows():
    s = [x for x in opto if x["key"].endswith(r.session)][0]
    g_ = s["grid"]
    mask = s["ctrl_matched_mask"][r.unit]
    occ = np.bincount(v.bin_index(g_["x"][(g_["dir"] == r.dir) & mask]), minlength=v.N_BINS)
    si_m.append(v.spatial_info(v.rate_map(s["spikes"][r.unit], g_, mask)[r.dir], occ))
k["si_ctrl_matched"] = si_m
k["boost_matched"] = k.si_light - k.si_ctrl_matched
for grp, kk in k.dropna(subset=["si_ctrl_matched"]).groupby("group"):
    r = stats.spearmanr(kk.si_ctrl_matched, kk.si_light)
    print(f"Variant, time-matched control, {grp}: n = {len(kk)}, SI matched control median {kk.si_ctrl_matched.median():.2f}, "
          f"rho = {r.statistic:.3f}, p = {r.pvalue:.2g}; increase median {kk.boost_matched.median():.2f}")
km = k.dropna(subset=["si_ctrl_matched"])
print("Variant, increase over time-matched control, unmasked against place: Kruskal-Wallis p = %.2g" %
      stats.kruskal(km[km.group == "unmasked"].boost_matched, km[km.group == "place"].boost_matched).pvalue)
k["ratio"] = k.si_light / k.si_ctrl
print("Variant, ratio light/control: unmasked median %.2f, place median %.2f, p = %.2g" % (
    k[k.group == "unmasked"].ratio.median(), k[k.group == "place"].ratio.median(),
    stats.kruskal(k[k.group == "unmasked"].ratio, k[k.group == "place"].ratio).pvalue))

**Result: partially reproduced.** Control and light spatial information are correlated for place
fields (rho = 0.566, p = 2e-06) but not significantly for unmasked fields (rho = 0.396, p = 0.093).
The increase from control to light is larger for unmasked fields (median 0.12 bits/spike) than for
place fields (-0.01; Kruskal-Wallis p = 0.0019). Against time-matched control maps, the increase is
negative in both groups (unmasked -0.13, place -0.11) and does not differ (p = 0.64). The larger increase thus
depends on comparing light maps with control maps built from more time.

## Claim fig4c-swr-cofiring-place-pairs

> Cofiring of place cell pairs during SPW-Rs in the home cage is positively correlated with their
> spatial overlap on the track (P < 1e-27), while pairs of a place cell and a non-place cell show
> no such relationship (P = 0.11).

In [ ]:
def ripple_counts(s):
    home = [(e["start"], e["stop"]) for e in s["epochs"] if e["behavioral_paradigm"] in ("BaselinePre", "BaselinePost")]
    r = s["ripples"]
    keep = np.zeros(len(r), bool)
    for a, b in home:
        keep |= (r[:, 0] >= a) & (r[:, 1] < b)
    r = r[keep]
    return np.array([np.searchsorted(x, r[:, 1]) - np.searchsorted(x, r[:, 0]) for x in s["spikes"]]), len(r)

pairs = []
for s in opto:
    counts, nrip = ripple_counts(s)
    cof = np.corrcoef(counts)
    s["cofiring"] = cof
    pyr = np.flatnonzero(s["pyr"])
    cmaps = [np.concatenate(s["ctrl_maps"][u]) for u in range(len(s["spikes"]))]
    lmaps = {u: np.concatenate(m) for u, m in s["light_maps"].items()}
    for i_, a in enumerate(pyr):
        for b in pyr[i_ + 1:]:
            kind = {2: "PC-PC", 1: "PC-nonPC", 0: "nonPC-nonPC"}[int(s["place"][a]) + int(s["place"][b])]
            pairs.append({"session": s["key"].split("_ses-")[1], "a": a, "b": b, "kind": kind, "cofiring": cof[a, b],
                          "overlap_ctrl": v.map_corr(cmaps[a], cmaps[b]),
                          "overlap_light": v.map_corr(lmaps[a], lmaps[b]) if (a in lmaps and b in lmaps) else np.nan,
                          "both_responsive": bool(s["responsive"][a] and s["responsive"][b])})
    print(s["key"].split("_ses-")[1], "home-cage ripples without stimulation:", nrip)
pr = pd.DataFrame(pairs)
for kind, k in pr.dropna(subset=["cofiring", "overlap_ctrl"]).groupby("kind"):
    r = stats.spearmanr(k.cofiring, k.overlap_ctrl)
    print(f"{kind}: n = {len(k)} pairs, Spearman rho = {r.statistic:.3f}, p = {r.pvalue:.2g}")
    for sess, kk in k.groupby("session"):
        rr = stats.spearmanr(kk.cofiring, kk.overlap_ctrl)
        print(f"    {sess}: n = {len(kk)}, rho = {rr.statistic:.3f}, p = {rr.pvalue:.2g}")
# Variant: cofiring as the correlation of participation (any spike) per ripple
for s in opto:
    counts, _ = ripple_counts(s)
    s["cofiring_bin"] = np.corrcoef((counts > 0).astype(float))
pr["cofiring_bin"] = [[x for x in opto if x["key"].endswith(r.session)][0]["cofiring_bin"][r.a, r.b] for r in pr.itertuples()]
for kind, k in pr.dropna(subset=["cofiring_bin", "overlap_ctrl"]).groupby("kind"):
    r = stats.spearmanr(k.cofiring_bin, k.overlap_ctrl)
    print(f"Variant, participation correlation, {kind}: n = {len(k)}, rho = {r.statistic:.3f}, p = {r.pvalue:.2g}")
print("Pairs dropped (a cell without spikes in ripples or a flat map):", int(pr[["cofiring", "overlap_ctrl"]].isna().any(axis=1).sum()))

**Result: not reproduced.** Over 961 place cell pairs, cofiring during home-cage ripples is not
correlated with spatial overlap (Spearman rho = 0.041, p = 0.21; per session 0.064 and 0.037). Over
887 pairs of a place cell and a non-place cell, the paper's test finds a correlation (rho = 0.112,
p = 0.00081), where the paper found none. Participation per ripple gives the same pattern (rho =
0.032, p = 0.32, and 0.116, p = 0.00051). The cofiring measure is this attempt's choice; the paper's
is in the supplement.

## Claim fig4d-light-unmasked-cofiring

> With light-induced spikes, the relationship between SPW-R cofiring and spatial overlap appears
> for pairs that include non-place cells (P < 1e-5, Spearman correlation).

In [ ]:
k = pr[(pr.kind == "PC-nonPC") & pr.both_responsive].dropna(subset=["cofiring", "overlap_light"])
r = stats.spearmanr(k.cofiring, k.overlap_light)
print(f"PC-nonPC pairs, both light-responsive, light maps: n = {len(k)}, Spearman rho = {r.statistic:.3f}, p = {r.pvalue:.2g}")
kc = pr[(pr.kind == "PC-nonPC") & pr.both_responsive].dropna(subset=["cofiring", "overlap_ctrl"])
rc = stats.spearmanr(kc.cofiring, kc.overlap_ctrl)
print(f"The same pairs with control maps: n = {len(kc)}, rho = {rc.statistic:.3f}, p = {rc.pvalue:.2g}")
for kind in ("PC-PC", "nonPC-nonPC"):
    kk = pr[(pr.kind == kind) & pr.both_responsive].dropna(subset=["cofiring", "overlap_light"])
    if len(kk) > 5:
        rr = stats.spearmanr(kk.cofiring, kk.overlap_light)
        print(f"Variant, {kind} pairs, both responsive, light maps: n = {len(kk)}, rho = {rr.statistic:.3f}, p = {rr.pvalue:.2g}")
for sess, kk in k.groupby("session"):
    if len(kk) > 5:
        rr = stats.spearmanr(kk.cofiring, kk.overlap_light)
        print(f"  {sess}: n = {len(kk)}, rho = {rr.statistic:.3f}, p = {rr.pvalue:.2g}")
print(f"\nRun time so far {(time.time() - started) / 60:.1f} min")

**Result: reproduced.** Over 290 pairs of a light-responsive place cell and a light-responsive
non-place cell, cofiring during ripples is correlated with the overlap of their light maps (rho =
0.124, p = 0.035). The correlation comes from sess9 (rho = 0.167, p = 0.008; sess12 rho = -0.132,
p = 0.42). The claim implies that the relationship needs light. For the same pairs with control maps
it is stronger (rho = 0.191, p = 0.0011), so that part does not hold here.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| cohort-counts | partially reproduced |
| fig1c-pulse-protocol | partially reproduced |
| fig1e-responsive-fraction | reproduced |
| fig1i-swr-decrease | reproduced |
| figs10-swr-gain-vs-baseline-rate | partially reproduced |
| fig2b-theta-trough-gain | not reproduced |
| figs11-place-cell-fraction | not reproduced |
| fig2e-infield-gain | reproduced |
| figs12-gain-rate-correlations | not reproduced |
| fig3a-place-cells-more-excitable | reproduced |
| fig3c-unmasked-fields | not reproduced |
| fig3e-ghost-fields | partially reproduced |
| fig3f-light-stability | not reproduced |
| fig3g-spatial-information | partially reproduced |
| fig4c-swr-cofiring-place-pairs | not reproduced |
| fig4d-light-unmasked-cofiring | reproduced |
| fig1j-light-intensity | blocked: data missing |
| fig1km-intracellular | blocked: data missing |
| figs9-excitatory-vs-inhibitory-swr | blocked: metadata insufficient |
| fig3h-position-decoding | blocked: methods underspecified |
| fig4ef-assemblies | blocked: methods underspecified |
| fig4gh-replay | blocked: methods underspecified |

How the analysis differs from the paper's. The Materials and Methods were not available. The
light-response test is the one in the first author's later code, the place fields use buzcode's
defaults, and the theta phase, cofiring and stability measures are common choices. Pyramidal cells
are units with a baseline rate below 5 Hz, as the files carry no cell type. Every claim on light
responses or place fields rests on 2 sessions of one mouse (91 putative pyramidal cells), the only
sessions with pulse times, against the paper's 822 cells. Control maps use all time without light;
the paper's control epochs were 20 ms intervals between pulses, which is shown as a variant for
Figures 3F and 3G.

What was found about the dandiset.

- Pulse times are in 2 of 28 sessions; 20 more sessions name uLED stimulation in their epochs but
  hold no pulse times, so their light responses cannot be computed. Laps are also in these 2 only.
- The pulse series are stated in watts but hold one arbitrary value per uLED (19 in one session,
  about 63000 in the other). They carry no light intensity, so the intensity results cannot be
  tested. The files hold series for 6 of the probe's 12 uLEDs.
- Every session is deposited twice, as a full file and a no-raw-data file with the same units,
  spikes, ripples and position. One full file lacks the session in its name. The full file of
  fCamk3 sess17 holds 378 s of raw signal and 6850 s of LFP of a 34621 s session, and six fCamk1
  full files have no raw signal.
- Units carry no cell type and the small files no waveforms.
- `session_start_time` is 2016-03-15 in every file, though the sessions are from 2020 and 2021.
- The linearized position has a conversion factor of 100 with values already in cm.
- Six sessions have epochs without paradigm or manipulation. 14 sessions include CNO or DMSO
  epochs, which the paper does not describe. The dandiset holds 28 sessions and 1351 putative
  pyramidal cells, where the paper reports 822 cells (about 19 sessions), and does not say which
  sessions the paper used.
- In three sessions the deposited ripple intervals mark population bursts weakly; fCamk2 has 0 to
  1110 ripples per session and no position.
- The file description of the pulses gives 40 to 60 ms offsets, the paper 20 to 40 ms.